# Phylogenetic trees from DNA distances

*bioinformatics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/phylogenetic_tree)

DNA sequences evolve down a hidden random tree, each branch one matrix product with the Jukes–Cantor transition matrix. From the leaves alone we count differences with one einsum over one-hot letters, correct them for hidden multiple substitutions, and rebuild the tree by UPGMA and by neighbour joining. Watch both trees as more sites are used, scored against the truth by the Robinson–Foulds distance.

**Vector form**

$$P(t) = \tfrac14\,\mathbf 1\mathbf 1^{\mathsf T} + \big(I - \tfrac14\,\mathbf 1\mathbf 1^{\mathsf T}\big)\,e^{-4t/3},\qquad d = -\tfrac34\ln\!\big(1 - \tfrac43\,p\big),\qquad Q = (n-2)\,D - \mathbf r\,\mathbf 1^{\mathsf T} - \mathbf 1\,\mathbf r^{\mathsf T}$$

**Index (tensor) form**

$$y_{l\beta} = x_{l\alpha}P_{\alpha\beta}(t),\qquad p_{ij} = 1 - \tfrac1L\,X_{il\alpha}X_{jl\alpha},\qquad Q_{ij} = (n-2)\,d_{ij} - d_{ik}\mathbf 1_k - d_{jk}\mathbf 1_k$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $P_{\alpha\beta}(t)$ | `jc_transition(t, device)` | (4, 4) | probability that letter $\alpha$ has become $\beta$ after a branch of length $t$ (expected substitutions per site) |
| $y_{rl\beta} = x_{rl\alpha}P_{\alpha\beta}$ | `einsum('rla,ab->rlb', x, P)` | (R, L, 4) | repeated $\alpha$ ⇒ summed: letter probabilities of the child at every site; sampled by inverse CDF |
| $X_{il\alpha}$ | `simulate(tree, n_sites, 1, ...)[0]` | (n, L, 4) | the observed leaf sequences, one-hot over A, C, G, T |
| $p_{ij} = 1 - X_{il\alpha}X_{jl\alpha}/L$ | `1 - einsum('ila,jla->ij', X, X) / L` | (n, n) | repeated $l, \alpha$ ⇒ summed: fraction of sites where $i$ and $j$ differ |
| $d_{ij}$ | `jukes_cantor(p)` | (n, n) | corrected distance $-\tfrac34\ln(1 - \tfrac43 p)$: counts the substitutions hidden by repeated hits at one site |
| $Q_{ij}$ | `nj_criterion(D)` | (n, n) | neighbour joining merges the pair with the smallest $Q$, not the smallest $d$; $r_i = d_{ik}\mathbf 1_k$ |
| $d_{ij} = M_{iv}b_v + M_{jv}b_v - 2M_{iv}b_vM_{jv}$ | `patristic(M, b)` | (n, n) | true path lengths; $M_{iv} = 1$ when branch $v$ lies between leaf $i$ and the root, $b_v$ its length |
| $\mathrm{RF}$ | `robinson_foulds(tree_a, tree_b)` | int | number of leaf bipartitions (one per inner branch) present in only one of the two trees |

The true tree is a random coalescent tree; with the relaxed clock every branch also gets its own rate, so leaves sit at different distances from the root. The raw fraction of differing sites $p$ saturates at $3/4$ and underestimates long paths (grey dots below the diagonal); the Jukes–Cantor $d$ is unbiased (blue dots on the diagonal) but noisier, and its noise shrinks as $1/\sqrt L$. UPGMA assumes all leaves are equally far from the root, so on a relaxed clock it can join two short branches that are not neighbours (red branches), however many sites it sees. Neighbour joining only needs the distances to add up along the tree and converges to the true unrooted shape; it is drawn rooted at the midpoint of its longest path, since distances alone cannot locate the root. Switch to a strict clock and UPGMA becomes consistent too.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/phylogenetic_tree/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import numpy as np

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import patristic


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    clock = params["clock"] == "strict"
    tree = tensors.random_tree(params["n_taxa"], params["tree_depth"], clock, seed=0)
    M, b = tensors.path_matrix(tree, dev)  # (n, V), (V,)
    d_true = patristic(M, b)  # (n, n) path lengths on the true tree
    X = tensors.simulate(tree, params["n_sites"], 1, seed=1, device=dev)[0]  # (n, L, 4)
    n, L = X.shape[0], X.shape[1]
    print(f"{n} taxa, {L} sites evolved under Jukes–Cantor along a {'clock-like' if clock else 'relaxed-clock'} tree")
    print(f"true tree: {len(solve.splits(tree))} inner branches, longest leaf-to-leaf path {float(d_true.max()):.2f}")

    n_frames = 4 if fast_mode() else 40
    sites = np.unique(np.geomspace(min(20, L), L, n_frames).astype(int))  # use the first `s` sites in each frame
    results = []
    for s in sites:
        p, d = solve.distances(X[:, :s])  # (n, n) each
        nj, up = solve.midpoint_root(solve.neighbour_joining(d)), solve.upgma(d)
        results.append((int(s), p, d, nj, up, solve.robinson_foulds(nj, tree), solve.robinson_foulds(up, tree)))
    for s, p, d, _, _, rf_nj, rf_up in results[:: max(1, len(results) // 5)] + results[-1:]:
        err_p = float((p - d_true).abs().mean())
        err_d = float((d - d_true).abs().mean())
        print(
            f"{s:5d} sites: mean |p − true| = {err_p:.3f}, |d_JC − true| = {err_d:.3f}, "
            f"RF(NJ) = {rf_nj}, RF(UPGMA) = {rf_up}"
        )
    worst = max(2 * n - 6, 1)
    print(f"Robinson–Foulds distance ranges from 0 (same unrooted shape) to {worst} for {n} taxa")
    return plot.animate(tree, results, d_true, out_dir / "phylogeny.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/phylogenetic_tree/equation.py
r"""Molecular evolution → tensor translation.

Jukes–Cantor model: every site mutates at the same rate to any of the other three letters. After a branch of length
t (expected substitutions per site) the letter-to-letter transition matrix is

    P_{\alpha\beta}(t) = 1/4 + (\delta_{\alpha\beta} − 1/4)\,e^{-4t/3}

so a child's letter probabilities are y_{l\beta} = x_{l\alpha} P_{\alpha\beta} (\alpha summed) for every site l at once.

Distances between n aligned sequences with one-hot letters X_{il\alpha} (taxon i, site l, letter \alpha):

    p_{ij} = 1 − X_{il\alpha} X_{jl\alpha} / L                 (l, \alpha summed: fraction of sites that differ)
    d_{ij} = −(3/4) \ln(1 − 4 p_{ij} / 3)                      Jukes–Cantor correction for hidden multiple hits

d estimates the path length between i and j on the tree; p saturates at 3/4 and underestimates long distances.

Tree building from d: UPGMA merges the closest pair and averages distances (assumes a molecular clock); neighbour
joining merges the pair minimising Q_{ij} = (n − 2) d_{ij} − r_i − r_j with r_i = d_{ij} 1_j (j summed), which
recovers the right tree for any additive distances, clock or not.
"""

import torch


def jc_transition(t: float, device: torch.device) -> torch.Tensor:
    r"""P_{αβ}(t) = 1/4 + (δ_{αβ} − 1/4) e^{−4t/3}: (4, 4), rows sum to 1."""
    eye = torch.eye(4, device=device)
    return 0.25 + (eye - 0.25) * torch.exp(torch.tensor(-4.0 * t / 3.0, device=device))


def evolve_branch(x: torch.Tensor, P: torch.Tensor) -> torch.Tensor:
    r"""y_{rlβ} = x_{rlα} P_{αβ}   (α summed): letter probabilities of the child at every site.   (R, L, 4)"""
    return torch.einsum("rla,ab->rlb", x, P)


def p_distance(X: torch.Tensor) -> torch.Tensor:
    r"""p_{ij} = 1 − X_{ilα} X_{jlα} / L   (l, α summed).   (..., n, L, 4) → (..., n, n)"""
    return 1.0 - torch.einsum("...ila,...jla->...ij", X, X) / X.shape[-2]


def jukes_cantor(p: torch.Tensor) -> torch.Tensor:
    r"""d_{ij} = −(3/4) ln(1 − 4 p_{ij}/3); p ≥ 3/4 (saturated) is capped just below."""
    return -0.75 * torch.log1p(-4.0 / 3.0 * p.clamp(max=0.749))


def nj_criterion(D: torch.Tensor) -> torch.Tensor:
    r"""Q_{ij} = (n − 2) D_{ij} − r_i − r_j with r_i = D_{ij} 1_j (j summed); diagonal set to +∞.   (n, n) → (n, n)"""
    n = D.shape[0]
    r = D.sum(dim=1)
    Q = (n - 2) * D - r[:, None] - r[None, :]
    return Q.fill_diagonal_(float("inf"))


def patristic(M: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    r"""Path lengths on a tree: d_{ij} = M_{iv} b_v + M_{jv} b_v − 2 M_{iv} b_v M_{jv}   (v summed over branches).

    M_{iv} = 1 when branch v (the edge above node v) lies on the path from leaf i to the root; b_v its length.
    (n, V), (V,) → (n, n)
    """
    depth = M @ b  # (n,) distance of every leaf from the root
    shared = torch.einsum("iv,v,jv->ij", M, b, M)  # length of the common root path of i and j
    return depth[:, None] + depth[None, :] - 2 * shared


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/phylogenetic_tree/tensors.py
"""Build the concrete tensors: a random true tree and DNA sequences evolved along it (synthetic, seeded)."""

from dataclasses import dataclass

import torch
import torch.nn.functional as F

from .equation import evolve_branch, jc_transition


@dataclass
class Tree:
    """Leaves are nodes 0 … n−1. parent[v] is v's parent (−1 for the root), length[v] the branch above v."""

    n_leaves: int
    parent: list[int]
    length: list[float]

    @property
    def root(self) -> int:
        return self.parent.index(-1)

    def children(self) -> list[list[int]]:
        kids: list[list[int]] = [[] for _ in self.parent]
        for v, p in enumerate(self.parent):
            if p >= 0:
                kids[p].append(v)
        return kids


def random_tree(n_taxa: int, depth: float, clock: bool, seed: int) -> Tree:
    """Coalescent tree: join two random lineages at exponentially spaced times, rescaled to root height `depth`.

    Without a clock each branch is then stretched by a random rate factor (log-normal), so leaves end up at
    different distances from the root (non-ultrametric).
    """
    gen = torch.Generator().manual_seed(seed)
    height = [0.0] * n_taxa
    parent = [-1] * (2 * n_taxa - 1)
    active, t = list(range(n_taxa)), 0.0
    while len(active) > 1:
        k = len(active)
        t += float(torch.empty(1).exponential_(k * (k - 1) / 2, generator=gen))
        pick = torch.randperm(k, generator=gen)[:2].tolist()
        a, b = active[pick[0]], active[pick[1]]
        new = len(height)
        height.append(t)
        parent[a] = parent[b] = new
        active = [v for v in active if v not in (a, b)] + [new]
    scale = depth / height[-1]
    length = [0.0 if p < 0 else (height[p] - height[v]) * scale for v, p in enumerate(parent)]
    if not clock:
        rates = torch.exp(0.7 * torch.randn(len(parent), generator=gen))
        length = [ell * float(r) for ell, r in zip(length, rates)]
    return Tree(n_taxa, parent, length)


def path_matrix(tree: Tree, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """M_{iv} = 1 if branch v is on the path from leaf i to the root: (n, V); branch lengths b_v: (V,)."""
    M = torch.zeros(tree.n_leaves, len(tree.parent), device=device)
    for i in range(tree.n_leaves):
        v = i
        while tree.parent[v] >= 0:
            M[i, v] = 1.0
            v = tree.parent[v]
    return M, torch.tensor(tree.length, device=device)


def simulate(tree: Tree, n_sites: int, n_rep: int, seed: int, device: torch.device) -> torch.Tensor:
    """Evolve R independent alignments down the tree under Jukes–Cantor: leaf sequences one-hot, (R, n, L, 4).

    Nodes are created bottom-up, so going through them in reverse order visits every parent before its children.
    """
    gen = torch.Generator().manual_seed(seed)
    seqs: dict[int, torch.Tensor] = {}
    root = tree.root
    seqs[root] = F.one_hot(torch.randint(0, 4, (n_rep, n_sites), generator=gen), 4).float().to(device)
    for v in range(len(tree.parent) - 1, -1, -1):
        if v == root:
            continue
        probs = evolve_branch(seqs[tree.parent[v]], jc_transition(tree.length[v], device))  # (R, L, 4)
        u = torch.rand(n_rep, n_sites, 1, generator=gen).to(device)
        letter = (u > probs.cumsum(dim=-1)).sum(dim=-1).clamp(max=3)  # inverse-CDF sampling at every site
        seqs[v] = F.one_hot(letter, 4).float()
    return torch.stack([seqs[i] for i in range(tree.n_leaves)], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/phylogenetic_tree/solve.py
"""Distances → trees (UPGMA and neighbour joining) and the Robinson–Foulds comparison of tree shapes."""

import torch

from .equation import jukes_cantor, nj_criterion, p_distance
from .tensors import Tree


def distances(X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """p-distance and Jukes–Cantor distance between all leaves. X: (n, L, 4) → two (n, n) matrices."""
    p = p_distance(X)
    d = jukes_cantor(p)
    return p, d.fill_diagonal_(0.0)


def _closest_pair(score: torch.Tensor, active: list[int]) -> tuple[int, int]:
    """Indices (into the full matrix) of the active pair with the smallest score."""
    sub = score[active][:, active].clone().fill_diagonal_(float("inf"))
    k = int(sub.argmin())
    return active[k // len(active)], active[k % len(active)]


def upgma(D: torch.Tensor) -> Tree:
    """Average-linkage clustering: merge the closest clusters, place them at height d/2 (a molecular clock)."""
    n = D.shape[0]
    D = D.clone()
    node = list(range(n))  # tree node held in each row of D
    size, height = [1] * n, [0.0] * n
    parent, length = [-1] * (2 * n - 1), [0.0] * (2 * n - 1)
    active = list(range(n))
    for new in range(n, 2 * n - 1):
        i, j = _closest_pair(D, active)
        h = float(D[i, j]) / 2
        for row in (i, j):
            parent[node[row]], length[node[row]] = new, max(0.0, h - height[row])
        merged = (size[i] * D[i] + size[j] * D[j]) / (size[i] + size[j])  # size-weighted average distance
        D[i], D[:, i] = merged, merged
        node[i], size[i], height[i] = new, size[i] + size[j], h
        active.remove(j)
    return Tree(n, parent, length)


def neighbour_joining(D: torch.Tensor) -> Tree:
    """Saitou–Nei: merge the pair minimising Q, give each its branch length, replace both by the new node u."""
    n = D.shape[0]
    D = D.clone()
    node = list(range(n))
    parent, length = [-1] * (2 * n - 2), [0.0] * (2 * n - 2)
    active = list(range(n))
    for new in range(n, 2 * n - 2):
        sub = D[active][:, active]
        Q = nj_criterion(sub)
        k = int(Q.argmin())
        a, b = k // len(active), k % len(active)
        i, j = active[a], active[b]
        r = sub.sum(dim=1)  # r_i = D_{ij} 1_j over the active nodes
        li = 0.5 * float(D[i, j]) + float(r[a] - r[b]) / (2 * (len(active) - 2))
        lj = float(D[i, j]) - li
        parent[node[i]], length[node[i]] = new, max(0.0, li)
        parent[node[j]], length[node[j]] = new, max(0.0, lj)
        du = 0.5 * (D[i] + D[j] - D[i, j])  # distance of the new node u to every other node
        D[i], D[:, i] = du, du
        D[i, i] = 0.0
        node[i] = new
        active.remove(j)
        if len(active) == 2:  # the last two nodes are joined by one branch: hang the leftover under u
            other = active[0] if active[1] == i else active[1]
            parent[node[other]], length[node[other]] = new, max(0.0, float(D[i, other]))
            break
    return Tree(n, parent, length)


def midpoint_root(tree: Tree) -> Tree:
    """Re-root an unrooted tree at the middle of its longest leaf-to-leaf path (for drawing only).

    The branch holding the midpoint is cut in two by a new root node; then the parent links are re-derived by a
    breadth-first walk from it. Leaves keep their numbers, so the splits do not change.
    """
    edges: dict[int, dict[int, float]] = {v: {} for v in range(len(tree.parent))}
    for v, p in enumerate(tree.parent):
        if p >= 0:
            edges[v][p] = edges[p][v] = tree.length[v]

    def walk(start: int):  # distance and previous node of every node, seen from `start`
        dist, prev, todo = {start: 0.0}, {start: -1}, [start]
        for u in todo:
            for w, ell in edges[u].items():
                if w not in dist:
                    dist[w], prev[w] = dist[u] + ell, u
                    todo.append(w)
        return dist, prev

    d0, _ = walk(0)
    a = max(range(tree.n_leaves), key=lambda v: d0[v])  # one end of the longest path
    da, prev = walk(a)
    b = max(range(tree.n_leaves), key=lambda v: da[v])  # the other end
    half, v = da[b] / 2, b
    while da[prev[v]] > half:  # climb from b towards a until the branch (prev[v], v) holds the midpoint
        v = prev[v]
    u, root = prev[v], len(tree.parent)
    del edges[u][v], edges[v][u]
    edges[root] = {u: half - da[u], v: da[v] - half}
    edges[u][root], edges[v][root] = edges[root][u], edges[root][v]
    _, parent = walk(root)
    length = [0.0 if parent[w] < 0 else edges[w][parent[w]] for w in range(root + 1)]
    return Tree(tree.n_leaves, [parent[w] for w in range(root + 1)], length)


def clades(tree: Tree) -> list[frozenset]:
    """Leaf set below every node (index = node)."""
    below: list[set] = [set() for _ in tree.parent]
    for leaf in range(tree.n_leaves):
        v = leaf
        while v >= 0:
            below[v].add(leaf)
            v = tree.parent[v]
    return [frozenset(s) for s in below]


def splits(tree: Tree) -> set[frozenset]:
    """Non-trivial bipartitions of the leaves made by cutting one branch, written as the side without leaf 0.

    The root is ignored, so rooted and unrooted trees with the same shape have the same splits.
    """
    n, out = tree.n_leaves, set()
    for clade in clades(tree):
        side = clade if 0 not in clade else frozenset(range(n)) - clade
        if 2 <= len(side) <= n - 2:
            out.add(side)
    return out


def robinson_foulds(a: Tree, b: Tree) -> int:
    """Number of splits found in only one of the two trees (0 = same unrooted topology)."""
    return len(splits(a) ^ splits(b))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/phylogenetic_tree/plot.py
"""Animate tree reconstruction as more sites are used: true tree, distances, NJ and UPGMA dendrograms."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

from .solve import clades, splits


def layout(tree):
    """x = distance from the root, y = leaf order (depth-first) and the mean of the children's y for inner nodes."""
    kids = tree.children()
    x, y = [0.0] * len(tree.parent), [0.0] * len(tree.parent)
    order: list[int] = []

    def visit(v: int, x0: float):
        x[v] = x0
        if not kids[v]:
            y[v] = len(order)
            order.append(v)
            return
        for c in kids[v]:
            visit(c, x0 + tree.length[c])
        y[v] = float(np.mean([y[c] for c in kids[v]]))

    visit(tree.root, 0.0)
    return x, y, kids


def draw(ax, tree, true_splits: set, title: str):
    """Rectangular dendrogram; branches whose split is not in the true tree are red."""
    ax.clear()
    x, y, kids = layout(tree)
    leaves = clades(tree)
    n = tree.n_leaves

    def wrong(v: int) -> bool:  # the branch above v makes a split that the true tree does not have
        side = leaves[v] if 0 not in leaves[v] else frozenset(range(n)) - leaves[v]
        return 2 <= len(side) <= n - 2 and side not in true_splits

    for v, cs in enumerate(kids):
        style = {"color": "#dc2626", "lw": 2.2} if wrong(v) else {"color": "#374151", "lw": 1.4}
        if cs:
            ax.plot([x[v], x[v]], [min(y[c] for c in cs), max(y[c] for c in cs)], **style)
        if tree.parent[v] >= 0:
            ax.plot([x[tree.parent[v]], x[v]], [y[v], y[v]], **style)
    span = max(x) or 1.0
    for leaf in range(n):
        ax.plot(x[leaf], y[leaf], "o", color="#2563eb", ms=4)
        ax.text(x[leaf] + 0.03 * span, y[leaf], f"t{leaf}", va="center", fontsize=8)
    ax.set_xlim(-0.03 * span, 1.18 * span)
    ax.set_ylim(n - 0.5, -0.5)
    ax.set_yticks([])
    ax.grid(False)
    ax.set_xlabel("substitutions per site", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_title(title, fontsize=9)


def animate(true_tree, results, d_true, out: Path) -> Path:
    """results: list of (sites, p, d, nj_tree, upgma_tree, rf_nj, rf_upgma); d_true (n, n) path lengths."""
    true_splits = splits(true_tree)
    iu = np.triu_indices(true_tree.n_leaves, 1)
    dt = d_true.cpu().numpy()[iu]

    fig, axes = plt.subplots(2, 2, figsize=(9, 6.4))
    ax_true, ax_d, ax_nj, ax_up = axes.ravel()
    draw(ax_true, true_tree, true_splits, "true tree (sequences evolved along it)")
    top = 1.15 * dt.max()
    ax_d.plot([0, top], [0, top], color="#9ca3af", ls="--", lw=1)
    sc_p = ax_d.scatter(dt, dt, s=12, color="#9ca3af", label="p-distance")
    sc_d = ax_d.scatter(dt, dt, s=12, color="#2563eb", label="Jukes–Cantor $d$")
    ax_d.set_xlim(0, top)
    ax_d.set_ylim(0, top)
    ax_d.set_xlabel("true path length on the tree", fontsize=8)
    ax_d.set_ylabel("estimated distance", fontsize=8)
    ax_d.legend(fontsize=7, loc="upper left")
    ax_d.tick_params(labelsize=7)
    head = ax_d.set_title(" ", fontsize=9)

    def update(k: int):
        sites, p, d, nj, up, rf_nj, rf_up = results[k]
        sc_p.set_offsets(np.c_[dt, p.cpu().numpy()[iu]])
        sc_d.set_offsets(np.c_[dt, d.cpu().numpy()[iu]])
        head.set_text(f"pairwise distances from {sites} sites")
        draw(ax_nj, nj, true_splits, f"neighbour joining: RF = {rf_nj}")
        draw(ax_up, up, true_splits, f"UPGMA: RF = {rf_up}  (red: wrong split)")

    update(0)
    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(results), 60), out, fps=4)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_taxa': 10,
    'n_sites': 3000,
    'tree_depth': 0.5,
    'clock': 'relaxed'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.phylogenetic_tree")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.phylogenetic_tree.main")
problem_dir = Path("problems/bioinformatics/phylogenetic_tree")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
